# 🔢 Notebook 2 — Embeddings + Indexação Qdrant

Implementa as Etapas 7 e 8 do pipeline:

- **Embeddings:** `intfloat/multilingual-e5-large-instruct` (1024 dims) — recomendação do PDF
- **Indexação:** Qdrant local com payload completo (texto_pai, ato_id, tipo_documento, situação)
- **Filtro nativo por tipo_documento** resolve o problema de distribuição (53% votos+notas)
- **Persistência em disco** — não recarrega vetores na memória entre execuções

**Pré-requisito:** Notebook 1 executado, `chunks_hierarquicos.parquet` no Drive

**⚠️ Ative GPU:** `Runtime > Change runtime type > T4 GPU` (~25min com GPU; ~8h em CPU)

In [ ]:
# ============================================================
# CÉLULA 1 — Instalação
# ============================================================
%%capture
!pip install sentence-transformers qdrant-client tqdm pyarrow huggingface_hub python-dotenv

In [ ]:
# ============================================================
# CÉLULA 2 — Imports e configs
# ============================================================
import os
import gc
import time
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, PayloadSchemaType,
)

# === TOKEN HUGGING FACE ===
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

print(f'HF_TOKEN: {"✅ configurado" if HF_TOKEN else "❌ não encontrado — configure nos Secrets"}')

# === CONFIGURAÇÕES ===
EMBEDDING_MODEL  = 'intfloat/multilingual-e5-large-instruct'
VECTOR_DIM       = 1024
BATCH_SIZE       = 64
COLLECTION_NAME  = 'aneel_legislacao'
HF_REPO          = 'JvPetas/aneel-legislacao'

QDRANT_PATH      = '/content/qdrant_storage'
EMB_CHECKPOINT   = '/content/embeddings_checkpoint.npy'
EMB_INDEX_FILE   = '/content/embeddings_idx.txt'
LOCAL_PARQUET    = '/content/chunks_hierarquicos.parquet'

# Verifica GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device.upper()}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('   ⚠️  Sem GPU — ative T4 em Runtime > Change runtime type')

HF_TOKEN: ✅ configurado
Device: CUDA
   GPU: NVIDIA A100-SXM4-40GB
   VRAM: 42.4 GB


In [ ]:
# ============================================================
# CÉLULA 3 — Carregar chunks do Hugging Face
# ============================================================
from huggingface_hub import hf_hub_download

print('📥 Baixando chunks do Hugging Face...')
path = hf_hub_download(
    repo_id='JvPetas/aneel-legislacao',
    filename='chunks_hierarquicos.parquet',
    repo_type='dataset',
    local_dir='/content/',
)
df_chunks = pd.read_parquet(path)
print(f'✅ {len(df_chunks):,} chunks carregados')
print(f'   Colunas: {list(df_chunks.columns)}')

📥 Baixando chunks do Hugging Face...


chunks_hierarquicos.parquet:   0%|          | 0.00/202M [00:00<?, ?B/s]

✅ 429,206 chunks carregados
   Colunas: ['chunk_id', 'chunk_pai_id', 'ato_id', 'tipo_documento', 'titulo', 'ementa', 'assunto', 'situacao', 'publicacao', 'autor', 'ano', 'contexto_juridico', 'numero_chunk', 'total_chunks', 'posicao_relativa', 'chunk_anterior_id', 'chunk_proximo_id', 'texto', 'texto_pai', 'aviso', 'arquivo_origem']


In [ ]:
# ============================================================
# CÉLULA 4 — Carrega modelo de embeddings
# ============================================================
print(f'📥 Carregando {EMBEDDING_MODEL}...')
print('   (~2GB de download na primeira vez)')
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
embed_model.max_seq_length = 384  # cobre os chunks filho com folga
print(f'✅ Modelo carregado | dim={embed_model.get_sentence_embedding_dimension()}')

# Validação de dimensionalidade
actual_dim = embed_model.get_sentence_embedding_dimension()
assert actual_dim == VECTOR_DIM, f'Dim esperado {VECTOR_DIM}, real {actual_dim}'

📥 Carregando intfloat/multilingual-e5-large-instruct...
   (~2GB de download na primeira vez)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

✅ Modelo carregado | dim=1024


/tmp/ipykernel_836/3912482490.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'✅ Modelo carregado | dim={embed_model.get_sentence_embedding_dimension()}')
/tmp/ipykernel_836/3912482490.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  actual_dim = embed_model.get_sentence_embedding_dimension()


In [ ]:
# ============================================================
# CÉLULA 5 — Gera embeddings (com checkpoint a cada 10k)
# ============================================================
# e5-large-instruct usa formato:
#   passage: <texto>      ← para documentos
#   query: Instruct: <task>\nQuery: <pergunta>  ← para queries (no notebook 3)

# Texto a indexar = contexto_juridico + texto_filho (enriquece embedding)
def make_passage(row):
    ctx = row['contexto_juridico']
    txt = row['texto']
    return f'passage: [{ctx}] {txt}' if ctx else f'passage: {txt}'

passages = df_chunks.apply(make_passage, axis=1).tolist()
total = len(passages)

# Verifica checkpoint
start_idx = 0
all_embs = []
if os.path.exists(EMB_CHECKPOINT) and os.path.exists(EMB_INDEX_FILE):
    saved = np.load(EMB_CHECKPOINT)
    with open(EMB_INDEX_FILE) as f:
        start_idx = int(f.read().strip())
    all_embs.append(saved)
    print(f'♻️  Retomando do checkpoint: {start_idx:,}/{total:,}')

print(f'🚀 Gerando embeddings para {total - start_idx:,} chunks (batch={BATCH_SIZE})')
t0 = time.time()

for i in tqdm(range(start_idx, total, BATCH_SIZE), desc='Embeddings', unit='batch'):
    batch = passages[i : i + BATCH_SIZE]
    with torch.no_grad():
        emb = embed_model.encode(
            batch,
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            show_progress_bar=False,
            convert_to_numpy=True,
        )
    all_embs.append(emb.astype('float32'))

    # Checkpoint a cada ~10k chunks (resiliência contra desconexão Colab)
    if (i // BATCH_SIZE) % max(1, 10_000 // BATCH_SIZE) == 0 and i > start_idx:
        np.save(EMB_CHECKPOINT, np.vstack(all_embs))
        with open(EMB_INDEX_FILE, 'w') as f:
            f.write(str(i + BATCH_SIZE))

embeddings = np.vstack(all_embs).astype('float32')
elapsed = (time.time() - t0) / 60
print(f'\n✅ Embeddings gerados em {elapsed:.1f} min')
print(f'   Shape: {embeddings.shape} | RAM: {embeddings.nbytes / 1e6:.1f} MB')

# Salvar embeddings finais
np.save('/content/embeddings_final.npy', embeddings)

# Liberar GPU
del embed_model
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

🚀 Gerando embeddings para 429,206 chunks (batch=64)


Embeddings:   0%|          | 0/6707 [00:00<?, ?batch/s]


✅ Embeddings gerados em 10.6 min
   Shape: (429206, 1024) | RAM: 1758.0 MB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CÉLULA 5-B — Salvar embeddings no Drive (persistência)
# ============================================================
from google.colab import drive
import shutil
import os

drive.mount('/content/drive', force_remount=True)

DRIVE_BACKUP = '/content/drive/MyDrive/aneel_rag'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

print('💾 Salvando embeddings no Drive...')
shutil.copy('/content/embeddings_final.npy', f'{DRIVE_BACKUP}/embeddings_final.npy')
print(f'✅ Salvo em {DRIVE_BACKUP}/embeddings_final.npy')

In [ ]:
# ============================================================
# CÉLULA 6 — Cria coleção Qdrant local
# ============================================================
# Qdrant em modo local — persiste em disco, sem servidor externo.
# Permite filtros por tipo_documento, situação, ano, ato_id durante o retrieval.

# Limpa storage antigo se existir (evita conflitos de schema)
if os.path.exists(QDRANT_PATH):
    os.system(f'rm -rf {QDRANT_PATH}')

client = QdrantClient(path=QDRANT_PATH)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
)

# Cria índices nos campos de filtro (acelera queries com where)
for field, schema in [
    ('tipo_documento', PayloadSchemaType.KEYWORD),
    ('situacao',       PayloadSchemaType.KEYWORD),
    ('ato_id',         PayloadSchemaType.KEYWORD),
    ('ano',            PayloadSchemaType.INTEGER),
]:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=field,
        field_schema=schema,
    )

print(f'✅ Coleção "{COLLECTION_NAME}" criada')
print(f'   Dimensões: {VECTOR_DIM} | Distância: COSINE')
print(f'   Índices: tipo_documento, situacao, ato_id, ano')

✅ Coleção "aneel_legislacao" criada
   Dimensões: 1024 | Distância: COSINE
   Índices: tipo_documento, situacao, ato_id, ano


/tmp/ipykernel_836/1675448356.py:25: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


In [ ]:
# ============================================================
# CÉLULA 6-B — Recarregar embeddings
# ============================================================
import numpy as np, os

DRIVE_BACKUP = '/content/drive/MyDrive/aneel_rag'

caminhos = [
    '/content/drive/MyDrive/embeddings_final.npy',
    f'{DRIVE_BACKUP}/embeddings_final.npy',
    '/content/embeddings_checkpoint.npy',
]

carregou = False
for caminho in caminhos:
    if os.path.exists(caminho):
        print(f'📥 Carregando de {caminho}...')
        embeddings = np.load(caminho)
        print(f'✅ Embeddings carregados: {embeddings.shape}')
        carregou = True
        break

if not carregou:
    print('❌ Nenhum arquivo encontrado. Rode a Célula 5 novamente.')

📥 Carregando de /content/embeddings_checkpoint.npy...
✅ Embeddings carregados: (419392, 1024)


In [ ]:
# ============================================================
# CÉLULA 7 — Upload dos vetores para o Qdrant
# ============================================================
# Insere em batches de 256 para eficiência

UPSERT_BATCH = 256

print(f'📤 Inserindo {len(embeddings):,} pontos no Qdrant...')
t0 = time.time()

for i in tqdm(range(0, len(embeddings), UPSERT_BATCH), desc='Upsert'):
    batch_end = min(i + UPSERT_BATCH, len(embeddings))
    points = []
    for j in range(i, batch_end):
        row = df_chunks.iloc[j]
        # Payload mínimo necessário para retrieval e citação
        payload = {
            'texto':            row['texto'],
            'texto_pai':        row['texto_pai'],
            'ato_id':           row['ato_id'],
            'tipo_documento':   row['tipo_documento'],
            'titulo':           row['titulo'],
            'ementa':           row['ementa'],
            'assunto':          row['assunto'],
            'situacao':         row['situacao'],
            'publicacao':       row['publicacao'],
            'autor':            row['autor'],
            'ano':              int(row['ano']),
            'contexto_juridico': row['contexto_juridico'],
            'arquivo_origem':   row['arquivo_origem'],
        }
        points.append(PointStruct(
            id=int(j),
            vector=embeddings[j].tolist(),
            payload=payload,
        ))

    client.upsert(collection_name=COLLECTION_NAME, points=points)

elapsed = (time.time() - t0) / 60
info = client.get_collection(COLLECTION_NAME)
print(f'\n✅ Upsert concluído em {elapsed:.1f} min')
print(f'   Pontos indexados: {info.points_count:,}')
print(f'   Status: {info.status}')

📤 Inserindo 419,392 pontos no Qdrant...


Upsert:   0%|          | 0/1639 [00:00<?, ?it/s]

/tmp/ipykernel_836/1201254743.py:38: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20224 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(collection_name=COLLECTION_NAME, points=points)



✅ Upsert concluído em 48.2 min
   Pontos indexados: 419,392
   Status: green


In [ ]:
# ============================================================
# CÉLULA 8 — Teste de retrieval rápido
# ============================================================
# Validação: carrega o modelo de novo (só pra teste) e roda 1 query

import os
from qdrant_client import QdrantClient

# Re-initialize Qdrant client if it's not already defined
if 'client' not in locals() or client is None:
    print('♻️ Re-initializing Qdrant client...')
    QDRANT_PATH      = '/content/qdrant_storage'
    client = QdrantClient(path=QDRANT_PATH)


print('🧪 Teste de retrieval...')
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=device)

# Formato query do e5-instruct
task = 'Given a question about Brazilian electric sector regulations, retrieve relevant legal passages'
query_text = 'Quais são os critérios para revisão tarifária das distribuidoras de energia?'
query_formatted = f'Instruct: {task}\nQuery: {query_text}'

q_emb = embed_model.encode(query_formatted, normalize_embeddings=True).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=q_emb,
    limit=5,
).points

print(f'\n🔍 Query: "{query_text}"')
for i, r in enumerate(results, 1):
    p = r.payload
    print(f"\n{i}. score={r.score:.4f} | {p['tipo_documento']} | {p['publicacao']}")
    print(f"   {p['ato_id']} | {p['titulo'][:80]}")
    print(f"   {p['texto'][:200]}...")

# Liberar memória
del embed_model
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

♻️ Re-initializing Qdrant client...


/tmp/ipykernel_836/2989478009.py:13: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <aneel_legislacao> contains 419392 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path=QDRANT_PATH)


🧪 Teste de retrieval...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


🔍 Query: "Quais são os critérios para revisão tarifária das distribuidoras de energia?"

1. score=0.9216 | anexo | 2021-02-08
   ren_912_2021 | REN - RESOLUÇÃO NORMATIVA 912/2021
    distribuição de
energia elétrica.
2. ABRANGÊNCIA
2. Aplica-se a todas as revisões e reajustes tarifários de concessionárias de serviço
público de distribuição de energia elétrica.
3. CRITÉRIOS GERAIS...

2. score=0.9184 | voto | 2016-02-01
   dsp_209_2016 | DSP - DESPACHO 209/2016
   ídas entre o último
Reajuste Tarifário Anual e a competência de setembro de 2015";
b. " Sobrecontratação / Exposição constituída em 2015";
c. "Neutralidade de Encargos Setoriais constituída entre o úl...

3. score=0.9153 | voto | 2016-02-01
   dsp_209_2016 | DSP - DESPACHO 209/2016
    a distribuidora pelo aumento
extraordinário dos custos incorridos (e por incorrer até o próximo evento tarifário) na aquisição da
energia elétrica e nos encargos setoriais.
13. Importante recordar qu...

4. score=0.9139 | texto_integral | 2021-

In [ ]:
# ============================================================
# CÉLULA 9 — Salvar Qdrant no Hugging Face
# ============================================================
from huggingface_hub import HfApi
import os, gc

# Token — compatível com Colab Secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

# Strip any whitespace from the token
if HF_TOKEN:
    HF_TOKEN = HF_TOKEN.strip()

if not HF_TOKEN:
    raise ValueError("❌ HF_TOKEN não encontrado — configure nos Secrets do Colab")

HF_REPO = 'JvPetas/aneel-legislacao'

# Fecha cliente com segurança
if 'client' in dir():
    client.close()
    del client
    gc.collect()

# Comprime storage
print('Comprimindo storage do Qdrant...')
tar_path = '/content/qdrant_storage.tar.gz'
ret = os.system(f'tar -czf {tar_path} -C /content qdrant_storage')
if ret != 0:
    raise RuntimeError("❌ Falha ao compactar qdrant_storage — verifique se a pasta existe")

size_mb = os.path.getsize(tar_path) / 1e6
print(f'✅ Compactado: {size_mb:.1f} MB')

# Upload
print('📤 Enviando para Hugging Face...')
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=tar_path,
    path_in_repo='qdrant_storage.tar.gz',
    repo_id=HF_REPO,
    repo_type='dataset',
)
print('✅ qdrant_storage.tar.gz publicado em JvPetas/aneel-legislacao')
print('\nNOTEBOOK 2 CONCLUÍDO!')
print('   Próximo: 03_rag_pipeline.ipynb')


Comprimindo storage do Qdrant...
✅ Compactado: 1292.7 MB
📤 Enviando para Hugging Face...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ent/qdrant_storage.tar.gz:   0%|          | 3.15MB / 1.29GB            

✅ qdrant_storage.tar.gz publicado em JvPetas/aneel-legislacao

NOTEBOOK 2 CONCLUÍDO!
   Próximo: 03_rag_pipeline.ipynb
